# 07. Ablation text trích từ ảnh & so sánh với LLM đa modal

Cần cache OCR + mô tả VLM từ `01b_image_text_extraction.ipynb`: trên Kaggle attach output của notebook đó làm input rồi sửa `IMAGE_TEXT_DIR`. Phần 2 cần GPU ~16GB (Kaggle T4 x2 hoặc P100) cho LLM 3B.

In [ ]:
# === Cell bootstrap (giống nhau ở mọi notebook, xem CODEBASE.md mục 6) ===
import os, subprocess, sys
from pathlib import Path

REPO = "github.com/yunaLee21/Multimodal_Sarcasm_Detection.git"
REF = "main"               # branch của mình, hoặc 1 commit hash để tái lặp kết quả
CODE_DIR = "/tmp/IE403"    # clone ra ngoài /kaggle/working để token không bị lưu vào output


def get_github_token():
    """Token cho repo private; repo public thì không cần tạo secret."""
    try:
        if os.environ.get("KAGGLE_KERNEL_RUN_TYPE"):
            from kaggle_secrets import UserSecretsClient
            return UserSecretsClient().get_secret("GITHUB_TOKEN")
        from google.colab import userdata
        return userdata.get("GITHUB_TOKEN")
    except Exception:
        return None


ON_CLOUD = bool(os.environ.get("KAGGLE_KERNEL_RUN_TYPE")) or "COLAB_RELEASE_TAG" in os.environ
if ON_CLOUD:
    if "COLAB_RELEASE_TAG" in os.environ and not os.environ.get("KAGGLE_KERNEL_RUN_TYPE"):  # ảnh Kaggle dựng từ ảnh Colab nên cũng có biến này
        from google.colab import drive
        drive.mount("/content/drive")
    token = get_github_token()
    url = f"https://{token}@{REPO}" if token else f"https://{REPO}"
    subprocess.run(["rm", "-rf", CODE_DIR])
    subprocess.run(["git", "clone", "-q", url, CODE_DIR], check=True)
    subprocess.run(["git", "-C", CODE_DIR, "checkout", "-q", REF], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{CODE_DIR}/requirements-kaggle.txt"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", CODE_DIR], check=True)
    if CODE_DIR + "/src" not in sys.path:
        sys.path.insert(0, CODE_DIR + "/src")  # kernel đang chạy chưa thấy package vừa cài -e
    os.chdir(CODE_DIR)
elif Path.cwd().name == "notebooks":
    os.chdir(Path.cwd().parent)  # local: chạy từ gốc repo (đã `pip install -e .` một lần)

print("cwd:", os.getcwd())
commit = subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip()
print("code commit:", commit or "unknown")

In [ ]:
import json
import logging
import matplotlib.pyplot as plt
from IPython.display import HTML, Image as IPImage, display

from vimmsd.utils.config import load_config

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s", force=True)
logging.getLogger("httpx").setLevel(logging.WARNING)

## 1. Ablation: đóng góp của OCR và mô tả VLM

Cùng mô hình cross-attention + focal loss, chỉ thay đổi text đưa vào segment thứ 2 của PhoBERT.

In [ ]:
cfg = load_config("configs/fusion_cross_attn_focal.yaml")
IMAGE_TEXT_DIR = None  # None = paths.cache_dir; Kaggle: vd. "/kaggle/input/<output-notebook-01b>/cache"
it_dir = [f"data.image_text.dir={IMAGE_TEXT_DIR}"] if IMAGE_TEXT_DIR else []

ABLATION = {
    "xattn_focal_caption": [],
    "xattn_focal_ocr": ["data.image_text.use_ocr=true", "data.max_length=256"],
    "xattn_focal_desc": ["data.image_text.use_description=true", "data.max_length=256"],
    "xattn_focal_ocr_desc": ["data.image_text.use_ocr=true", "data.image_text.use_description=true",
                             "data.max_length=256"],
}
for name, overrides in ABLATION.items():
    args = " ".join([f"name={name}", *overrides, *it_dir])
    !python scripts/train.py --config configs/fusion_cross_attn_focal.yaml --override {args} --note "{name}"

In [ ]:
import pandas as pd
rows = []
for name in ABLATION:
    m = json.loads((Path(cfg.paths.output_dir) / name / "test_metrics.json").read_text())
    rows.append({"thí nghiệm": name, "macro-F1": m["macro_f1"],
                 **{f"F1 {k}": v["f1"] for k, v in m["per_class"].items()}})
pd.DataFrame(rows).set_index("thí nghiệm").round(4)

## 2. LLM đa modal zero-shot / few-shot

Chạy trên tập test đã chia (có nhãn). Output thô lưu JSONL, chạy tiếp được nếu bị ngắt.

In [ ]:
import torch
from vimmsd.data.dataset import load_all_records
from vimmsd.analysis.llm_compare import (HFVisionLanguageClassifier, compare_predictions,
                                         evaluate_llm, select_few_shot_examples)
from vimmsd.training.metrics import format_metrics

labels = list(cfg.data.labels)
records = load_all_records(cfg)
test_records = records["test"]
out_dir = Path(cfg.paths.output_dir) / "llm"
out_dir.mkdir(parents=True, exist_ok=True)

llm = HFVisionLanguageClassifier("Qwen/Qwen2.5-VL-3B-Instruct", labels=labels)

In [ ]:
zs_metrics, zs_rows = evaluate_llm(llm, test_records, labels, out_path=out_dir / "qwen_zero_shot.jsonl")
print("zero-shot | tỉ lệ câu trả lời không hợp lệ:", zs_metrics["invalid_rate"])
print(format_metrics(zs_metrics))

In [ ]:
examples = select_few_shot_examples(records["train"], labels, k_per_class=1)
fs_metrics, fs_rows = evaluate_llm(llm, test_records, labels, examples=examples,
                                   out_path=out_dir / "qwen_few_shot.jsonl")
print("few-shot | tỉ lệ câu trả lời không hợp lệ:", fs_metrics["invalid_rate"])
print(format_metrics(fs_metrics))

## 3. So sánh với model fine-tune

In [ ]:
!python scripts/evaluate.py --config configs/fusion_cross_attn_focal.yaml --split test
import pandas as pd
pred = pd.read_csv(Path(cfg.paths.output_dir) / cfg.name / "test_predictions.csv", dtype={"id": str})
model_out = {"ids": pred.id.tolist(), "preds": [labels.index(p) for p in pred.pred]}
cmp = compare_predictions(fs_rows, model_out, labels)
display(pd.crosstab(cmp.label, cmp.case))
cmp["caption"] = cmp.id.map({r["id"]: r["caption"] for r in test_records})
cmp[cmp.case == "only_llm"][["label", "llm_pred", "model_pred", "caption"]].head(10)

In [ ]:
cmp[cmp.case == "only_model"][["label", "llm_pred", "model_pred", "caption"]].head(10)